# 05. HistGradientBoosting

Este notebook entrena exclusivamente un `HistGradientBoostingClassifier` reproducible sobre los splits de `invoice-risk-v1`. El preprocessing se ajusta sólo con train y luego se aplica a validation y test.

## Configuración requerida

Configure MLflow como se documenta en el README mediante `MLFLOW_TRACKING_URI`. No escriba URLs ni credenciales en este notebook. Entregue el contexto académico vigente mediante `INVOICEOPS_ORGANIZATION_SLUG`, `INVOICEOPS_OWNER_TYPE`, `INVOICEOPS_OWNER_ID` y `INVOICEOPS_CREATED_BY_RUT`.

In [ ]:
import csv
import os
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from invoiceops_ml.data import MODEL_FEATURES, TARGET

DATASET_DIR = Path(os.environ.get("INVOICEOPS_DATASET_DIR", "data/invoice-risk-v1"))
NUMERIC_FEATURE_INDEXES = tuple(range(7))
CATEGORICAL_FEATURE_INDEXES = (7,)


def load_split(name: str) -> tuple[list[list[object]], list[int]]:
    with (DATASET_DIR / f"{name}.csv").open(newline="", encoding="utf-8") as file:
        rows = list(csv.DictReader(file))
    features = [
        [
            int(row["invoice_amount_cents"]),
            int(row["vendor_tenure_days"]),
            int(row["previous_incidents_12m"]),
            float(row["amount_vs_vendor_median"]),
            int(row["has_purchase_order"] == "True"),
            int(row["three_way_match"] == "True"),
            int(row["bank_account_recently_changed"] == "True"),
            row["country_risk"],
        ]
        for row in rows
    ]
    target = [int(row[TARGET] == "True") for row in rows]
    return features, target


train_features, train_target = load_split("train")
validation_features, validation_target = load_split("validation")
test_features, test_target = load_split("test")
len(train_target), len(validation_target), len(test_target), MODEL_FEATURES

## Entrenamiento y evaluación

`ColumnTransformer` conserva las features numéricas y codifica `country_risk` dentro del `Pipeline`. El encoder produce una matriz densa porque `HistGradientBoostingClassifier` no acepta matrices dispersas. Como el pipeline se ajusta sólo con train, no hay leakage desde validation ni test.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", "passthrough", NUMERIC_FEATURE_INDEXES),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURE_INDEXES),
    ]
)
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", HistGradientBoostingClassifier(max_iter=200, random_state=202605)),
    ]
)
model.fit(train_features, train_target)


def classification_metrics(
    prefix: str, features: list[list[object]], target: list[int]
) -> dict[str, float]:
    predictions = model.predict(features)
    return {
        f"{prefix}_accuracy": accuracy_score(target, predictions),
        f"{prefix}_precision": precision_score(target, predictions, zero_division=0),
        f"{prefix}_recall": recall_score(target, predictions, zero_division=0),
        f"{prefix}_f1": f1_score(target, predictions, zero_division=0),
    }


metrics = {
    **classification_metrics("validation", validation_features, validation_target),
    **classification_metrics("test", test_features, test_target),
}
metrics

## Registro en MLflow

El run registra parámetros, métricas, tags de ownership y el artifact `model` con el pipeline completo.

In [ ]:
from tempfile import TemporaryDirectory

import mlflow
import mlflow.sklearn

from invoiceops_ml.mlflow import configure_mlflow, mlflow_config_from_env
from invoiceops_ml.ownership import (
    OwnershipContext,
    select_owner_experiment,
    set_run_ownership_tags,
)


def required_environment(name: str) -> str:
    value = os.environ.get(name, "").strip()
    if not value:
        raise ValueError(f"{name} must be set from the current InvoiceOps context")
    return value


configure_mlflow(mlflow_config_from_env())
ownership_context = OwnershipContext(
    organization_slug=required_environment("INVOICEOPS_ORGANIZATION_SLUG"),
    owner_type=required_environment("INVOICEOPS_OWNER_TYPE"),
    owner_id=required_environment("INVOICEOPS_OWNER_ID"),
    created_by_rut=required_environment("INVOICEOPS_CREATED_BY_RUT"),
)
select_owner_experiment(ownership_context)

with mlflow.start_run(run_name="hist-gradient-boosting") as run:
    set_run_ownership_tags(ownership_context)
    mlflow.log_params(model.get_params())
    mlflow.log_metrics(metrics)
    with TemporaryDirectory() as model_dir:
        mlflow.sklearn.save_model(model, model_dir)
        mlflow.log_artifacts(model_dir, artifact_path="model")

run.info.run_id

## Revisión

Abra el run en la UI de MLflow y confirme parámetros, métricas `validation_*` y `test_*`, los tags de ownership y el artifact `model`. El siguiente notebook recuperará los runs existentes para compararlos sin reentrenar.